In [ ]:
from langchain_core.documents import Document
# create a simple text file
import os
from langchain_community.document_loaders import TextLoader,PyPDFLoader,PyMuPDFLoader,CSVLoader
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from pathlib import Path
# Directory Loader
from langchain_community.document_loaders import DirectoryLoader
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
from langchain_core.messages import HumanMessage, SystemMessage
import time
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List,Dict,Any,Tuple
from langchain_community.retrievers import BM25Retriever
import pickle

C:\Users\G P INTERNATIONAL\AppData\Local\Temp\ipykernel_8424\3843211636.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader,PyPDFLoader,PyMuPDFLoader,CSVLoader


In [2]:
def process_all_files(directory_path):
    all_documents= []
    dir_path = Path(directory_path).resolve() # Resolves relative paths cleanly

    if not dir_path.is_dir():
        raise ValueError(f"The path provided '{directory_path}' is not a valid directory.")

    # find all pdf files recursively
    extensions= ("**/*.pdf","**/*.csv","**/*.txt","**/*.md")
    file_list= []
    for ext in extensions:
        file_list.extend(list(dir_path.glob(ext)))
    print(f"found {len(file_list)} total files to process")


    # Initialize the structural splitter for Markdown files
    headers_to_split_on = [
        ("#", "Header_1"),
        ("##", "Header_2"),
        ("###", "Header_3"),
        ("####", "Header_4"),
    ]

    md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)

    for file in file_list:
        ext= file.suffix.lower()
        print(f"Processing ({ext}): {file.name}")
        print(f"Processing: {file.name}")

        try:
            if ext=='.pdf':
                loader= PyPDFLoader(str(file))
                documents= loader.load()

            elif ext=='.csv':
                # CSVLoader turn every row into its own Document object
                loader= CSVLoader(str(file),encoding='utf-8')
                documents= loader.load()

            elif ext=='.txt':
                loader= TextLoader(str(file),encoding='utf-8')
                documents= loader.load()

            elif ext=='.md':
                # not to use TextLoader, read raw text and use Markdown splitter immediately
                with open(file,'r',encoding='utf-8') as f:
                    text= f.read()
                documents= md_splitter.split_text(text)
            
            else:
                print(f"Unsupported file type skipped: ({file.name})")
                continue

            # metadata

            for doc in documents:
                doc.metadata['source'] = file.name
                doc.metadata['source_file']= file.name
                doc.metadata['file_type']= ext.replace('.','')

            all_documents.extend(documents)
            print(f"Successfully loaded {len(documents)} objects from {file.name}")
        
        except Exception as e:
            print(f"Error loading file {file.name} : {str(e)}")

    print(f"total documens loaded: {len(all_documents)}")
    return all_documents

In [4]:
documents= process_all_files("./data")

found 106 total files to process
Processing (.pdf): Answers.pdf
Processing: Answers.pdf
Successfully loaded 3 objects from Answers.pdf
Processing (.pdf): Chapter 1 - Chemical Reactions and Equations.pdf
Processing: Chapter 1 - Chemical Reactions and Equations.pdf
Successfully loaded 16 objects from Chapter 1 - Chemical Reactions and Equations.pdf
Processing (.pdf): Chapter 10 - The Human Eye and the Colourful World.pdf
Processing: Chapter 10 - The Human Eye and the Colourful World.pdf
Successfully loaded 10 objects from Chapter 10 - The Human Eye and the Colourful World.pdf
Processing (.pdf): Chapter 11 - Electricity.pdf
Processing: Chapter 11 - Electricity.pdf
Successfully loaded 24 objects from Chapter 11 - Electricity.pdf
Processing (.pdf): Chapter 12 - Magnetic Effects of Electric Current.pdf
Processing: Chapter 12 - Magnetic Effects of Electric Current.pdf
Successfully loaded 13 objects from Chapter 12 - Magnetic Effects of Electric Current.pdf
Processing (.pdf): Chapter 13 - Our 

In [5]:
cache_file= "bm25_index.pkl"

In [6]:
chunks_dict= {}
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap= chunk_overlap,
        length_function= len,
        separators= ["\n\n","\n"," ",""]
    )

    docs_to_split= []
    final_chunks= []

    for doc in documents:
        if doc.metadata.get('file_type')=='csv':
            final_chunks.append(doc)
        else:
            # PDFs, TXT, and structurally-parsed Markdown go here for length enforcement
            docs_to_split.append(doc)

    if docs_to_split:
        final_chunks.extend(text_splitter.split_documents(docs_to_split))
    print(f"length of documents: {len(documents)} into {len(final_chunks)} chunks")

    if final_chunks:
        for chunk in final_chunks:
            chunk.metadata['chunk_id']= str(uuid.uuid4())
            chunks_dict[chunk.metadata['chunk_id']]= chunk
            
            page_content= chunk.page_content
            meta= chunk.metadata

            breadcrumbs= ""

            if "Header_1" in meta: breadcrumbs+= f" -> {meta['Header_1']}"
            if "Header_2" in meta: breadcrumbs+= f" -> {meta['Header_2']}"
            if "Header_3" in meta: breadcrumbs+= f" -> {meta['Header_3']}"
            if breadcrumbs: breadcrumbs+= "\n"
            
            page_content= breadcrumbs+ page_content
            chunk.page_content=page_content
        print("example chunk: ")
        print(f"content: {final_chunks[0].page_content[:200]}...")
        print(f"Metadata: {final_chunks[0].metadata}")

    return final_chunks

In [7]:
chunks= split_documents(documents)
chunks

length of documents: 1982 into 5771 chunks
example chunk: 
content: Answ er s
Chapter 1
1. (i) 2. (d) 3. (a)
Chapter 2
1. (d) 2. (b) 3. (d) 4. (c)
Chapter 3
1. (d) 2. (c) 3. (a) 4. (c)
Chapter 4
1. (b) 2. (c) 3. (b)
Chapter 5
1. (c) 2. (a) 3. (d) 4. (b)
Chapter 6
1. (...
Metadata: {'producer': 'GPL Ghostscript 8.15', 'creator': 'PageMaker 7.0', 'creationdate': '2022-08-17T16:43:14+00:00', 'author': 'admin', 'moddate': '2026-03-24T12:00:46+05:30', 'title': 'Answer.pmd', 'source': 'Answers.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Answers.pdf', 'file_type': 'pdf', 'chunk_id': '892fc3de-be36-44c4-b5c2-4b240ec8cf53'}


[Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PageMaker 7.0', 'creationdate': '2022-08-17T16:43:14+00:00', 'author': 'admin', 'moddate': '2026-03-24T12:00:46+05:30', 'title': 'Answer.pmd', 'source': 'Answers.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Answers.pdf', 'file_type': 'pdf', 'chunk_id': '892fc3de-be36-44c4-b5c2-4b240ec8cf53'}, page_content='Answ er s\nChapter 1\n1. (i) 2. (d) 3. (a)\nChapter 2\n1. (d) 2. (b) 3. (d) 4. (c)\nChapter 3\n1. (d) 2. (c) 3. (a) 4. (c)\nChapter 4\n1. (b) 2. (c) 3. (b)\nChapter 5\n1. (c) 2. (a) 3. (d) 4. (b)\nChapter 6\n1. (d) 2. (b) 3. (d)\nChapter 7\n1. (b) 2. (c) 3. (d)\nChapter 8\n1. (c)\nChapter 9\n1. (d) 2. (d) 3. (b)\n4. (a) 5. (d) 6. (c)\n7. Distance less than 15 cm; virtual; Enlarged.\n9. Yes\n10. 16.7 cm from the lens on the other side; 3.3 cm, reduced; real, inverted.\n11. 30 cm\n12. 6.0 cm, behind the mirror; virtual, erect\n13. m = 1 indicates that image formed by a plane mirror is of the sa

In [17]:
print(chunks[0].page_content)

Answ er s
Chapter 1
1. (i) 2. (d) 3. (a)
Chapter 2
1. (d) 2. (b) 3. (d) 4. (c)
Chapter 3
1. (d) 2. (c) 3. (a) 4. (c)
Chapter 4
1. (b) 2. (c) 3. (b)
Chapter 5
1. (c) 2. (a) 3. (d) 4. (b)
Chapter 6
1. (d) 2. (b) 3. (d)
Chapter 7
1. (b) 2. (c) 3. (d)
Chapter 8
1. (c)
Chapter 9
1. (d) 2. (d) 3. (b)
4. (a) 5. (d) 6. (c)
7. Distance less than 15 cm; virtual; Enlarged.
9. Yes
10. 16.7 cm from the lens on the other side; 3.3 cm, reduced; real, inverted.
11. 30 cm
12. 6.0 cm, behind the mirror; virtual, erect
13. m = 1 indicates that image formed by a plane mirror is of the same size as the object.
Further, the positive sign of m indicates that the image is virtual and erect.
14. 8.6 cm, behind the mirror; virtual, erect; 2.2 cm, reduced.
15. 54 cm on the object side; 14 cm, magnified; real, inverted.
16. –0.50 m; concave lens
17. + 0.67 m; converging lens
Reprint 2026-27


In [19]:
print(chunks[4].page_content)

has taken place? Let us perform some activities to find the answer to
these questions.
Figure 1.1
Burning of a magnesium ribbon in air and collection of magnesium
oxide in a watch-glass
Activity 1.1Activity 1.1Activity 1.1Activity 1.1Activity 1.1
CAUTION: This Activity needs
the teacher’s assistance. It
would be better if students
wear suitable eyeglasses.
/square6Clean a magnesium ribbon
about 3-4 cm long by rubbing
it with sandpaper .
/square6Hold it with a pair of tongs.
Burn it using a spirit lamp or
burner and collect the ash so
formed in a watch-glass as
shown in Fig. 1.1. Burn  the
magnesium ribbon keeping it
away as far as possible from
your eyes.
/square6What do you observe?
“Facts are not science — as the dictionary is not literature.”
Martin H. Fischer
Reprint 2026-27
